# Model traing preparation and training

## Imports

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

from pathlib import Path

from model.Model import MiniFCOSFaceV1
from dataset.Dataset import WiderFaceDataset, detection_collate
from generatetargets import generate_targets

## Datasets and data loaders

### Train

In [ ]:
train_dataset = WiderFaceDataset(
    image_root= "../dataset/WIDER_train/images",
    annotation_file= "../dataset/wider_face_split/wider_face_train_bbx_gt.txt",
    image_size=320
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=detection_collate
)

### Validate

In [ ]:
val_dataset = WiderFaceDataset(
    image_root= "../dataset/WIDER_train/images",
    annotation_file= "../dataset/wider_face_split/wider_face_train_bbx_gt.txt",
    image_size=320
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=detection_collate
)

## Loss function

In [ ]:
import torch.nn.functional as F

def binary_focal_loss_with_logits(
    logits,
    targets,
    alpha=0.25,
    gamma=2.0
):
    """
    logits: raw model output
    targets: 0 ili 1
    """

    bce_loss = F.binary_cross_entropy_with_logits(
        logits,
        targets,
        reduction="none"
    )

    probabilities = torch.sigmoid(logits)

    p_t = (
        probabilities * targets +
        (1.0 - probabilities) * (1.0 - targets)
    )

    alpha_t = (
        alpha * targets +
        (1.0 - alpha) * (1.0 - targets)
    )

    focal_loss = alpha_t * (1.0 - p_t).pow(gamma) * bce_loss

    return focal_loss.mean()

## Detection loss

In [ ]:
def detection_loss(predictions, targets):
    """
    predictions: [B, 6, 40, 40], raw CNN output
    targets:     [B, 6, 40, 40], targets from generate_targets()
    """

    face_logits = predictions[:, 0]
    bbox_predictions = F.softplus(predictions[:, 1:5])
    centerness_logits = predictions[:, 5]

    face_targets = targets[:, 0]
    bbox_targets = targets[:, 1:5]
    centerness_targets = targets[:, 5]

    positive_mask = face_targets == 1
    positive_count = positive_mask.sum()

    face_loss = binary_focal_loss_with_logits(
        face_logits,
        face_targets
    )

    if positive_count > 0:
        bbox_predictions = bbox_predictions.permute(0, 2, 3, 1)
        bbox_targets = bbox_targets.permute(0, 2, 3, 1)

        bbox_loss = F.smooth_l1_loss(
            bbox_predictions[positive_mask],
            bbox_targets[positive_mask],
            beta=0.1
        )

        centerness_loss = F.binary_cross_entropy_with_logits(
            centerness_logits[positive_mask],
            centerness_targets[positive_mask]
        )
    else:
        bbox_loss = predictions.sum() * 0.0
        centerness_loss = predictions.sum() * 0.0

    total_loss = (
        face_loss +
        5.0 * bbox_loss +
        centerness_loss
    )

    return total_loss, {
        "total": total_loss.detach().item(),
        "face": face_loss.detach().item(),
        "bbox": bbox_loss.detach().item(),
        "centerness": centerness_loss.detach().item(),
        "positive_count": int(positive_count.item())
    }

## Optimizer, training running on GPU

In [ ]:
import torch
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MiniFCOSFaceV1().to(device)

optimizer = optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

number_of_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Device:", device)
print("Parameters:", number_of_parameters)

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()

    loss_sums = {
        "total": 0.0,
        "face": 0.0,
        "bbox": 0.0,
        "centerness": 0.0
    }

    batch_count = 0

    for batch_index, (images, boxes_list) in enumerate(loader):
        images = images.to(device, non_blocking=True)

        targets = torch.stack([
            generate_targets(boxes.to(device))
            for boxes in boxes_list
        ])

        predictions = model(images)

        loss, loss_info = detection_loss(predictions, targets)

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=10.0
        )

        optimizer.step()

        for key in loss_sums:
            loss_sums[key] += loss_info[key]

        batch_count += 1

        if (batch_index + 1) % 100 == 0:
            print(
                f"Batch {batch_index + 1}/{len(loader)} | "
                f"loss: {loss_info['total']:.4f} | "
                f"face: {loss_info['face']:.4f} | "
                f"bbox: {loss_info['bbox']:.4f} | "
                f"center: {loss_info['centerness']:.4f} | "
                f"pozitivnih: {loss_info['positive_count']}"
            )

    return {
        key: value / batch_count
        for key, value in loss_sums.items()
    }


@torch.no_grad()
def validate_one_epoch(model, loader, device):
    model.eval()

    loss_sums = {
        "total": 0.0,
        "face": 0.0,
        "bbox": 0.0,
        "centerness": 0.0
    }

    batch_count = 0

    for images, boxes_list in loader:
        images = images.to(device, non_blocking=True)

        targets = torch.stack([
            generate_targets(boxes.to(device))
            for boxes in boxes_list
        ])

        predictions = model(images)

        _, loss_info = detection_loss(predictions, targets)

        for key in loss_sums:
            loss_sums[key] += loss_info[key]

        batch_count += 1

    return {
        key: value / batch_count
        for key, value in loss_sums.items()
    }

## Training loop

In [ ]:
checkpoint_dir = Path("checkpoints")
checkpoint_dir.mkdir(exist_ok=True)

num_epochs = 20
best_val_loss = float("inf")

history = {
    "train_total": [],
    "val_total": [],
    "train_face": [],
    "val_face": [],
    "train_bbox": [],
    "val_bbox": [],
    "train_centerness": [],
    "val_centerness": []
}

for epoch in range(num_epochs):
    print("\n" + "=" * 70)
    print(f"Epoch {epoch + 1}/{num_epochs}")
    print("=" * 70)

    train_metrics = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        device=device
    )

    val_metrics = validate_one_epoch(
        model=model,
        loader=val_loader,
        device=device
    )

    history["train_total"].append(train_metrics["total"])
    history["val_total"].append(val_metrics["total"])

    history["train_face"].append(train_metrics["face"])
    history["val_face"].append(val_metrics["face"])

    history["train_bbox"].append(train_metrics["bbox"])
    history["val_bbox"].append(val_metrics["bbox"])

    history["train_centerness"].append(train_metrics["centerness"])
    history["val_centerness"].append(val_metrics["centerness"])

    print("\nProsjek epohe:")
    print(
        f"Train | total: {train_metrics['total']:.4f} | "
        f"face: {train_metrics['face']:.4f} | "
        f"bbox: {train_metrics['bbox']:.4f} | "
        f"center: {train_metrics['centerness']:.4f}"
    )
    print(
        f"Val   | total: {val_metrics['total']:.4f} | "
        f"face: {val_metrics['face']:.4f} | "
        f"bbox: {val_metrics['bbox']:.4f} | "
        f"center: {val_metrics['centerness']:.4f}"
    )

    checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_metrics": train_metrics,
        "val_metrics": val_metrics,
        "center_sampling_ratio": 1.0,
        "image_size": 320,
        "feature_size": 40
    }

    epoch_path = checkpoint_dir / f"minifcos_face_v1_epoch_{epoch + 1:02d}.pt"
    torch.save(checkpoint, epoch_path)

    if val_metrics["total"] < best_val_loss:
        best_val_loss = val_metrics["total"]

        best_path = checkpoint_dir / "minifcos_face_v1_best.pt"
        torch.save(checkpoint, best_path)

        print("\nSpremljen novi najbolji model:", best_path)
        
    print("Spremljen checkpoint:", epoch_path)

## Loss graph

In [ ]:
from pathlib import Path
import re
import torch
import matplotlib.pyplot as plt

checkpoint_dir = Path("checkpoints")

checkpoint_paths = sorted(
    checkpoint_dir.glob("minifcos_face_v1_epoch_*.pt"),
    key=lambda path: int(re.search(r"epoch_(\d+)", path.name).group(1))
)

epochs = []
train_total_loss = []
val_total_loss = []

for path in checkpoint_paths:
    checkpoint = torch.load(path, map_location="cpu")

    epochs.append(checkpoint["epoch"])
    train_total_loss.append(checkpoint["train_metrics"]["total"])
    val_total_loss.append(checkpoint["val_metrics"]["total"])

plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    train_total_loss,
    marker="o",
    linewidth=2,
    label="Gubitak na skupu za treniranje"
)

plt.plot(
    epochs,
    val_total_loss,
    marker="o",
    linewidth=2,
    label="Gubitak na validacijskom skupu"
)

plt.xlabel("Epoha")
plt.ylabel("Ukupni gubitak")
plt.title("Promjena ukupnog gubitka tijekom treniranja modela")
plt.xticks(epochs)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig("loss_graf.png", dpi=300, bbox_inches="tight")
plt.show()